In [1]:
import json
import pandas as pd

In [2]:
DATA = "../data"
TELEMETRY_FILE = f"{DATA}/farm/tractor_telemetry.csv.gz"
S4_FILE = f"{DATA}/sda/scintillation_series.json"
LABELS_FILE = f"{DATA}/labels/train_episodes.json"

TRACTORS = ["T1", "T2", "T3"]
ONE_MIN = pd.Timedelta(minutes=1)

## Look at the data first

In [3]:
tele = pd.read_csv(TELEMETRY_FILE, parse_dates=["ts"])
tele.shape

(777600, 11)


In [4]:
tele.head()

                         ts tractor_id        lat        lon  speed_mps  cross_track_error_m  num_sats  mean_cn0_dbhz  min_cn0_dbhz  fix_state  base_link_ok
0 2026-05-04 00:00:00+00:00         T1  46.900041 -97.203944        4.5                0.009        12           45.1          40.4  RTK_FIXED          True
1 2026-05-04 00:00:01+00:00         T1  46.900081 -97.203944        4.5                0.012        11           45.7          39.7  RTK_FIXED          True
2 2026-05-04 00:00:02+00:00         T1  46.900121 -97.203944        4.5                0.004        13           45.8          40.3  RTK_FIXED          True
3 2026-05-04 00:00:03+00:00         T1  46.900162 -97.203944        4.5                0.008        12           45.0          40.7  RTK_FIXED          True
4 2026-05-04 00:00:04+00:00         T1  46.900202 -97.203944        4.5                0.019        13           45.8          41.1  RTK_FIXED          True


In [5]:
tele.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 777600 entries, 0 to 777599
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype              
---  ------               --------------   -----              
 0   ts                   777600 non-null  datetime64[ns, UTC]
 1   tractor_id           777600 non-null  object             
 2   lat                  777600 non-null  float64            
 3   lon                  777600 non-null  float64            
 4   speed_mps            777600 non-null  float64            
 5   cross_track_error_m  777600 non-null  float64            
 6   num_sats             777600 non-null  int64              
 7   mean_cn0_dbhz        777600 non-null  float64            
 8   min_cn0_dbhz         777600 non-null  float64            
 9   fix_state            777600 non-null  object             
 10  base_link_ok         777600 non-null  bool               
dtypes: bool(1), datetime64[ns, UTC](1), float64(6), int64(1), object(

In [6]:
tele.describe().round(2)

             lat        lon  speed_mps  cross_track_error_m   num_sats  mean_cn0_dbhz  min_cn0_dbhz
count  777600.00  777600.00  777600.00            777600.00  777600.00      777600.00     777600.00
mean       46.90     -97.20       3.00                 0.22      11.61          42.34         36.57
std         0.00       0.00       2.12                 0.93       1.28           5.53          7.04
min        46.90     -97.20       0.00                -0.09       3.00           4.90         -6.00
25%        46.90     -97.20       0.00                 0.00      11.00          44.00         38.10
50%        46.90     -97.20       4.50                 0.01      12.00          44.80         39.70
75%        46.90     -97.20       4.50                 0.03      13.00          45.30         40.60
max        46.91     -97.19       4.50                26.82      13.00          48.00         45.70


In [7]:
# any missing values? which time range per tractor?
print("missing values:", tele.isna().sum().sum())
tele.groupby("tractor_id").ts.agg(["min", "max", "count"])

missing values: 0


                                 min                       max   count
tractor_id                                                            
T1         2026-05-04 00:00:00+00:00 2026-05-06 23:59:59+00:00  259200
T2         2026-05-04 00:00:00+00:00 2026-05-06 23:59:59+00:00  259200
T3         2026-05-04 00:00:00+00:00 2026-05-06 23:59:59+00:00  259200


In [8]:
tele.fix_state.value_counts()

fix_state
RTK_FIXED     622997
DGPS           71722
RTK_FLOAT      71562
AUTONOMOUS      8434
NO_FIX          2885
Name: count, dtype: int64


In [9]:
# speed is either 4.5 (working) or 0 (parked). Which UTC hours do they work?
print(tele.speed_mps.value_counts())
sorted(tele[tele.speed_mps > 1].ts.dt.hour.unique())

speed_mps
4.5    518400
0.0    259200
Name: count, dtype: int64


[np.int32(0), np.int32(1), np.int32(2), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23)]


In [10]:
# what does a normal working hour look like? (May 4, 15:00-16:00, nothing labeled there)
quiet = tele[(tele.ts >= "2026-05-04 15:00") & (tele.ts < "2026-05-04 16:00")]
quiet[["mean_cn0_dbhz", "min_cn0_dbhz", "num_sats", "cross_track_error_m"]].median()

mean_cn0_dbhz          45.000
min_cn0_dbhz           40.100
num_sats               12.000
cross_track_error_m     0.014
dtype: float64


In [11]:
# S4 = storm sensor, one reading every 15 minutes
s4 = pd.DataFrame(json.load(open(S4_FILE))["series"])
s4["ts"] = pd.to_datetime(s4.ts, utc=True)
s4.s4.describe().round(3)

count    288.000
mean       0.151
std        0.210
min        0.020
25%        0.044
50%        0.056
75%        0.080
max        0.795
Name: s4, dtype: float64


In [12]:
# when is S4 high?
high = s4[s4.s4 >= 0.2]
high.groupby(high.ts.dt.date).ts.agg(["min", "max"])

                                 min                       max
ts                                                            
2026-05-05 2026-05-05 07:00:00+00:00 2026-05-05 21:45:00+00:00
2026-05-06 2026-05-06 13:00:00+00:00 2026-05-06 15:45:00+00:00


In [13]:
# the 12 known answers for May 4-5
labels = pd.DataFrame(json.load(open(LABELS_FILE))["episodes"])
labels["start"] = pd.to_datetime(labels.start)
labels["end"] = pd.to_datetime(labels.end)
labels[["id", "class", "tractors", "start", "end"]]

        id            class      tractors                     start                       end
0   TRN-01    SPACE_WEATHER  [T1, T2, T3] 2026-05-05 11:00:00+00:00 2026-05-05 16:00:00+00:00
1   TRN-02    SPACE_WEATHER  [T1, T2, T3] 2026-05-05 19:00:00+00:00 2026-05-05 22:00:00+00:00
2   TRN-03          JAMMING  [T1, T2, T3] 2026-05-04 16:30:00+00:00 2026-05-04 17:12:00+00:00
3   TRN-04          JAMMING      [T1, T3] 2026-05-04 21:00:00+00:00 2026-05-04 21:36:00+00:00
4   TRN-05          JAMMING  [T1, T2, T3] 2026-05-05 20:00:00+00:00 2026-05-05 20:36:00+00:00
5   TRN-06         SPOOFING          [T2] 2026-05-04 13:12:00+00:00 2026-05-04 13:54:00+00:00
6   TRN-07         SPOOFING          [T1] 2026-05-05 00:30:00+00:00 2026-05-05 01:18:00+00:00
7   TRN-08         SPOOFING          [T3] 2026-05-05 21:30:00+00:00 2026-05-05 22:06:00+00:00
8   TRN-09  EQUIPMENT_FAULT          [T2] 2026-05-04 12:00:00+00:00 2026-05-04 14:00:00+00:00
9   TRN-10  EQUIPMENT_FAULT          [T1] 2026-05-05 02:00:0

What I take from this:
- Normal while working: signal about 45, weakest satellite about 40, 12 satellites, drift about 0.02 m.
- Tractors work 11:00-03:00 UTC and are parked the rest. Parked rows must be ignored.
- S4 is high on May 5 and on May 6 13:00-16:00. On May 5 it starts at 07:00, but the label starts at 11:00, when the tractors start working.

**Idea: a tractor has a problem when it looks different from what it should look like right now.**
"Should" = the other two tractors + the storm sensor.

## One row per minute, one column per tractor

In [14]:
tele["xte"] = tele.cross_track_error_m.abs()       # drift left or right both count
tele["link"] = tele.base_link_ok.astype(float)

def per_minute(column, how):
    return tele.pivot_table(index=tele.ts.dt.floor("1min"), columns="tractor_id", values=column, aggfunc=how)

signal  = per_minute("mean_cn0_dbhz", "mean")   # average signal strength
weakest = per_minute("min_cn0_dbhz", "min")     # weakest satellite
drift   = per_minute("xte", "mean")             # metres off the planned line
link    = per_minute("link", "mean")            # share of the minute the correction link was up
working = per_minute("speed_mps", "mean") > 1

minutes = signal.index
s4_now = s4.set_index("ts").s4.reindex(minutes, method="ffill")   # latest S4 reading at each minute
signal.shape

(4320, 3)


In [15]:
# joins flagged minutes into (start, end) episodes
# max_gap: bridge gaps up to this many minutes, min_length: drop shorter episodes
def to_episodes(flag, max_gap, min_length):
    episodes = []
    for minute in flag.index[flag]:
        if episodes and minute - episodes[-1][1] <= max_gap * ONE_MIN:
            episodes[-1][1] = minute + ONE_MIN
        else:
            episodes.append([minute, minute + ONE_MIN])
    return [(a, b) for a, b in episodes if b - a >= min_length * ONE_MIN]

detections = []
def report(start, end, cause, tractors):
    detections.append({"start": start, "end": end, "class": cause, "tractors": tractors})

## 1. Space weather
**Rule:** all 3 working tractors have signal below 40 at the same minute, and S4 is 0.2 or higher.

**Example (TRN-02, May 5 19:00):** all three go from 45 to 35 in the same minute and S4 jumps from 0.06 to 0.42.
A storm hits the whole region, so every tractor drops by the same amount.

In [16]:
look = signal.loc["2026-05-05 18:58":"2026-05-05 19:02"].round(1)
look["s4"] = s4_now
look

tractor_id                   T1    T2    T3     s4
ts                                                
2026-05-05 18:58:00+00:00  45.0  45.0  45.0  0.057
2026-05-05 18:59:00+00:00  45.0  45.0  45.0  0.057
2026-05-05 19:00:00+00:00  35.5  35.2  35.5  0.418
2026-05-05 19:01:00+00:00  35.5  35.5  35.2  0.418
2026-05-05 19:02:00+00:00  35.3  35.4  35.5  0.418


In [17]:
fleet_signal = signal.median(axis=1)
storm_minute = working.all(axis=1) & (s4_now >= 0.2) & (fleet_signal < 40)

storms = to_episodes(storm_minute, max_gap=5, min_length=30)
for start, end in storms:
    report(start, end, "SPACE_WEATHER", TRACTORS)
    print(start, "to", end)

2026-05-05 11:00:00+00:00 to 2026-05-05 16:00:00+00:00
2026-05-05 19:00:00+00:00 to 2026-05-05 22:00:00+00:00
2026-05-06 13:00:00+00:00 to 2026-05-06 16:00:00+00:00


## 2. Jamming
**Rule:** a tractor's signal is 4 dB or more below normal. Normal = 45 in calm weather. During a storm, normal = that storm's own level.

**Example (TRN-03, May 4 16:30):** S4 is calm, but signal falls to 22 / 32 / 27 in one minute.
Each tractor drops by a different amount because each is a different distance from the jammer.
A tractor that drops less than 4 dB on average is left out (T2 in TRN-04).

In [18]:
signal.loc["2026-05-04 16:28":"2026-05-04 16:32"].round(1)

tractor_id                   T1    T2    T3
ts                                         
2026-05-04 16:28:00+00:00  45.0  45.0  45.1
2026-05-04 16:29:00+00:00  44.9  45.1  45.0
2026-05-04 16:30:00+00:00  22.0  31.9  26.6
2026-05-04 16:31:00+00:00  15.7  31.2  25.1
2026-05-04 16:32:00+00:00  27.2  32.6  28.9


In [19]:
calm_level = signal[working & (s4_now < 0.2).values[:, None]].stack().median()

normal = pd.Series(calm_level, index=minutes)
for start, end in storms:
    in_storm = (minutes >= start) & (minutes < end)
    normal[in_storm] = fleet_signal[in_storm].median()

drop = -signal.sub(normal, axis=0)              # dB below normal, per tractor
jammed = working & (drop >= 4)

jam_minute = pd.Series(False, index=minutes)    # checks 3 and 4 skip these minutes
for start, end in to_episodes(jammed.any(axis=1), max_gap=3, min_length=5):
    span = (minutes >= start) & (minutes < end)
    hit = [t for t in TRACTORS if drop.loc[span, t].mean() >= 4]
    if hit:
        report(start, end, "JAMMING", hit)
        jam_minute[span] = True
        print(start, "to", end, hit)

2026-05-04 16:30:00+00:00 to 2026-05-04 17:12:00+00:00 ['T1', 'T2', 'T3']
2026-05-04 21:00:00+00:00 to 2026-05-04 21:36:00+00:00 ['T1', 'T3']
2026-05-05 20:00:00+00:00 to 2026-05-05 20:36:00+00:00 ['T1', 'T2', 'T3']
2026-05-06 02:12:00+00:00 to 2026-05-06 02:54:00+00:00 ['T1', 'T2', 'T3']
2026-05-06 18:30:00+00:00 to 2026-05-06 19:00:00+00:00 ['T1', 'T2', 'T3']


## 3. Spoofing
**Rule:** one tractor drifts over 1 m more than the other two, while its signal is within 2 dB of theirs.

**Example (TRN-06, May 4 13:12):** T2's drift climbs 0.08, 0.35, 0.64, 0.92 ... up to 6 m. T1 and T3 stay at 0.02.
The signal looks healthy, which is what makes it spoofing and not jamming.
The drift grows slowly, so after finding it I walk the start back to where drift first passed 0.15 m.

In [20]:
drift.loc["2026-05-04 13:10":"2026-05-04 13:20"].iloc[::2].round(2)

tractor_id                   T1    T2    T3
ts                                         
2026-05-04 13:10:00+00:00  0.02  0.02  0.02
2026-05-04 13:12:00+00:00  0.02  0.08  0.01
2026-05-04 13:14:00+00:00  0.01  0.35  0.02
2026-05-04 13:16:00+00:00  0.03  0.64  0.03
2026-05-04 13:18:00+00:00  0.01  0.92  0.02
2026-05-04 13:20:00+00:00  0.02  1.21  0.02


In [21]:
extra_drift = drift.sub(drift.median(axis=1), axis=0)          # drift compared with the other tractors
signal_like_peers = signal.sub(fleet_signal, axis=0).abs() < 2

for t in TRACTORS:
    ok = working[t] & signal_like_peers[t] & ~jam_minute
    for start, end in to_episodes(ok & (extra_drift[t] > 1.0), max_gap=2, min_length=5):
        while ok.get(start - ONE_MIN, False) and extra_drift[t][start - ONE_MIN] > 0.15:
            start -= ONE_MIN
        while ok.get(end, False) and extra_drift[t][end] > 0.15:
            end += ONE_MIN
        report(start, end, "SPOOFING", [t])
        print(start, "to", end, t)

2026-05-05 00:31:00+00:00 to 2026-05-05 01:18:00+00:00 T1
2026-05-06 16:31:00+00:00 to 2026-05-06 17:12:00+00:00 T1
2026-05-04 13:13:00+00:00 to 2026-05-04 13:54:00+00:00 T2
2026-05-05 21:34:00+00:00 to 2026-05-05 22:06:00+00:00 T3
2026-05-06 21:49:00+00:00 to 2026-05-06 22:24:00+00:00 T3


## 4. Equipment fault
**Rule:** only one tractor is bad and the other two are fine. Two kinds:
- **Antenna:** its weakest satellite is 8 dB or more below the fleet's.
- **Link:** its correction link is up less than 98% of the minute while the others' links are up.

**Antenna example (TRN-09, May 4 12:00):** T2's weakest satellite is 12-19 while T1 and T3 read 27-38. T2's average signal barely moves, so only the weakest-satellite column shows it.

**Link example (TRN-10, May 5 02:00):** T1's link drops to about 50% in some minutes (about 70% on average). T2 and T3 stay at 100%.

In [22]:
print(weakest.loc["2026-05-04 12:06":"2026-05-04 12:10"].round(1))
link.loc["2026-05-05 01:58":"2026-05-05 02:03"].round(2)

tractor_id                   T1    T2    T3
ts                                         
2026-05-04 12:06:00+00:00  31.2  11.8  29.0
2026-05-04 12:07:00+00:00  37.1  18.7  37.5
2026-05-04 12:08:00+00:00  38.1  18.9  37.8
2026-05-04 12:09:00+00:00  38.1  17.2  37.5
2026-05-04 12:10:00+00:00  37.0  20.3  38.6


tractor_id                   T1   T2   T3
ts                                       
2026-05-05 01:58:00+00:00  1.00  1.0  1.0
2026-05-05 01:59:00+00:00  1.00  1.0  1.0
2026-05-05 02:00:00+00:00  0.47  1.0  1.0
2026-05-05 02:01:00+00:00  0.48  1.0  1.0
2026-05-05 02:02:00+00:00  0.92  1.0  1.0
2026-05-05 02:03:00+00:00  0.98  1.0  1.0


In [23]:
weakest_vs_fleet = weakest.sub(weakest.median(axis=1), axis=0)

for t in TRACTORS:
    others_link_up = link.drop(columns=t).min(axis=1) >= 0.98
    antenna = working[t] & ~jam_minute & (weakest_vs_fleet[t] <= -8)
    bad_link = working[t] & ~jam_minute & (link[t] < 0.98) & others_link_up
    for start, end in to_episodes(antenna | bad_link, max_gap=5, min_length=10):
        report(start, end, "EQUIPMENT_FAULT", [t])
        print(start, "to", end, t)

2026-05-05 02:00:00+00:00 to 2026-05-05 02:48:00+00:00 T1
2026-05-06 01:00:00+00:00 to 2026-05-06 02:00:00+00:00 T1
2026-05-04 12:00:00+00:00 to 2026-05-04 14:00:00+00:00 T2
2026-05-06 19:48:00+00:00 to 2026-05-06 20:29:00+00:00 T2


## 5. Multipath
**Rule:** each pass through the north end of the field where the weakest satellite drops 4 dB or more compared with the minute before.

**Example (TRN-11, May 4 14:04:18):** T3's weakest satellite falls from 40 to about 32 for 17 seconds while it turns at the north end, then recovers. Average signal does not move.
It happens at the same place on every pass, so it is a reflection off something there, not an attack. This check works per second, not per minute.

In [24]:
t3 = tele[tele.tractor_id == "T3"].set_index("ts")
t3.loc["2026-05-04 14:04:12":"2026-05-04 14:04:38", ["lat", "mean_cn0_dbhz", "min_cn0_dbhz"]].iloc[::3]

                                 lat  mean_cn0_dbhz  min_cn0_dbhz
ts                                                               
2026-05-04 14:04:12+00:00  46.906645           45.3          39.8
2026-05-04 14:04:15+00:00  46.906767           44.7          39.7
2026-05-04 14:04:18+00:00  46.906888           45.2          33.5
2026-05-04 14:04:21+00:00  46.907010           44.7          34.2
2026-05-04 14:04:24+00:00  46.907131           44.7          28.8
2026-05-04 14:04:27+00:00  46.907145           45.4          34.0
2026-05-04 14:04:30+00:00  46.907023           44.5          35.2
2026-05-04 14:04:33+00:00  46.906902           44.8          34.6
2026-05-04 14:04:36+00:00  46.906780           45.6          40.8


In [25]:
north_strip = tele.lat.max() - 0.00035          # last ~40 m of the field

for t in TRACTORS:
    one = tele[tele.tractor_id == t].set_index("ts")
    in_strip = (one.lat >= north_strip) & (one.speed_mps > 1)
    pass_id = (in_strip != in_strip.shift()).cumsum()
    for _, visit in one[in_strip].groupby(pass_id[in_strip]):
        start, end = visit.index[0], visit.index[-1] + pd.Timedelta(seconds=1)
        before = one.min_cn0_dbhz[start - ONE_MIN : start - pd.Timedelta(seconds=1)].median()
        if before - visit.min_cn0_dbhz.median() >= 4:
            report(start, end, "MULTIPATH", [t])

found = pd.DataFrame(detections).sort_values("start").reset_index(drop=True)
found["class"].value_counts()

class
MULTIPATH          1450
SPOOFING              5
JAMMING               5
EQUIPMENT_FAULT       4
SPACE_WEATHER         3
Name: count, dtype: int64


## Check against the 12 known answers (May 4-5)

In [26]:
rows = []
for _, lab in labels.iterrows():
    match = found[(found["class"] == lab["class"]) & (found.start < lab.end) & (found.end > lab.start)
                  & found.tractors.apply(lambda ts: set(ts) == set(lab.tractors))]
    ours = match.iloc[0] if len(match) else None
    rows.append({
        "id": lab.id, "cause": lab["class"], "tractors": ",".join(lab.tractors),
        "true start": lab.start.strftime("%d %H:%M:%S"),
        "our start": ours.start.strftime("%d %H:%M:%S") if ours is not None else "MISSED",
        "true end": lab.end.strftime("%H:%M:%S"),
        "our end": ours.end.strftime("%H:%M:%S") if ours is not None else "",
    })
pd.DataFrame(rows)

        id            cause  tractors   true start    our start  true end   our end
0   TRN-01    SPACE_WEATHER  T1,T2,T3  05 11:00:00  05 11:00:00  16:00:00  16:00:00
1   TRN-02    SPACE_WEATHER  T1,T2,T3  05 19:00:00  05 19:00:00  22:00:00  22:00:00
2   TRN-03          JAMMING  T1,T2,T3  04 16:30:00  04 16:30:00  17:12:00  17:12:00
3   TRN-04          JAMMING     T1,T3  04 21:00:00  04 21:00:00  21:36:00  21:36:00
4   TRN-05          JAMMING  T1,T2,T3  05 20:00:00  05 20:00:00  20:36:00  20:36:00
5   TRN-06         SPOOFING        T2  04 13:12:00  04 13:13:00  13:54:00  13:54:00
6   TRN-07         SPOOFING        T1  05 00:30:00  05 00:31:00  01:18:00  01:18:00
7   TRN-08         SPOOFING        T3  05 21:30:00  05 21:34:00  22:06:00  22:06:00
8   TRN-09  EQUIPMENT_FAULT        T2  04 12:00:00  04 12:00:00  14:00:00  14:00:00
9   TRN-10  EQUIPMENT_FAULT        T1  05 02:00:00  05 02:00:00  02:48:00  02:48:00
10  TRN-11        MULTIPATH        T3  04 14:04:18  04 14:04:18  14:04:35  1

In [27]:
# false alarms: the labels have 10 non-multipath episodes on May 4-5
dev = found[(found.start < "2026-05-06") & (found["class"] != "MULTIPATH")]
len(dev)

10


## May 6 (the scored day)

In [28]:
may6 = found[found.start >= "2026-05-06"]
print("multipath dips:", (may6["class"] == "MULTIPATH").sum(), " total detections:", len(may6), "(limit 500)")
may6[may6["class"] != "MULTIPATH"]

multipath dips: 486  total detections: 493 (limit 500)


                         start                       end            class      tractors
1004 2026-05-06 01:00:00+00:00 2026-05-06 02:00:00+00:00  EQUIPMENT_FAULT          [T1]
1041 2026-05-06 02:12:00+00:00 2026-05-06 02:54:00+00:00          JAMMING  [T1, T2, T3]
1129 2026-05-06 13:00:00+00:00 2026-05-06 16:00:00+00:00    SPACE_WEATHER  [T1, T2, T3]
1235 2026-05-06 16:31:00+00:00 2026-05-06 17:12:00+00:00         SPOOFING          [T1]
1296 2026-05-06 18:30:00+00:00 2026-05-06 19:00:00+00:00          JAMMING  [T1, T2, T3]
1336 2026-05-06 19:48:00+00:00 2026-05-06 20:29:00+00:00  EQUIPMENT_FAULT          [T2]
1400 2026-05-06 21:49:00+00:00 2026-05-06 22:24:00+00:00         SPOOFING          [T3]
